# Preliminary Exploratory Data Analysis

Auto-generated overview of a CSV dataset. Drop your file in `data/raw/` and run all cells.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
%matplotlib inline

## 1. Data Loading

Detects and loads the first CSV found in `data/raw/`.

In [ ]:
raw_dir = Path.cwd().parent / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

csv_files = list(raw_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {raw_dir}. Place a CSV file there and rerun.")

file_path = csv_files[0]
print(f"Loading: {file_path.name}")

df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

In [ ]:
df.head(10)

In [ ]:
df.tail(5)

## 2. Column Overview

Data types, non-null counts, and memory usage.

In [ ]:
info_buffer = pd.io.common.StringIO()
df.info(buf=info_buffer, verbose=True, show_counts=True)
print(info_buffer.getvalue())

In [ ]:
dtype_counts = df.dtypes.value_counts()
print("Column type distribution:")
for dtype, count in dtype_counts.items():
    print(f"  {dtype.name}: {count} columns")

## 3. Missing Values Analysis

In [ ]:
# Per-column nulls
missing_cols = pd.DataFrame({
    "Missing": df.isna().sum(),
    "%": (df.isna().sum() / len(df) * 100).round(2)
}).sort_values("Missing", ascending=False)

missing_cols = missing_cols[missing_cols["Missing"] > 0]
if missing_cols.empty:
    print("No missing values found.")
else:
    display(missing_cols)
    print(f"\nTotal missing values: {df.isna().sum().sum():,}")

In [ ]:
if not missing_cols.empty:
    plt.figure(figsize=(12, max(4, len(missing_cols) * 0.3)))
    sns.heatmap(df.isna(), cbar=False, cmap="viridis", yticklabels=False)
    plt.title("Missing Values Heatmap")
    plt.xlabel("Columns")
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-row null distribution
nulls_per_row = df.isna().sum(axis=1)

print("Rows with nulls:")
for threshold in [1, 2, 5, 10, 20, 50]:
    count = (nulls_per_row >= threshold).sum()
    print(f"  ≥{threshold:>3} nulls: {count:>6,} rows ({count / len(df) * 100:>5.1f}%)")

print(f"\nNulls per row — distribution:")
dist = nulls_per_row.value_counts().sort_index()
display(dist.head(20).to_frame("# rows"))

In [ ]:
# Row null histogram
if nulls_per_row.max() > 0:
    plt.figure(figsize=(10, 4))
    sns.histplot(nulls_per_row, bins=min(50, nulls_per_row.max() + 1), discrete=True)
    plt.title("Distribution of Nulls per Row")
    plt.xlabel("Number of nulls in row")
    plt.ylabel("Row count")
    plt.tight_layout()
    plt.show()

In [ ]:
# Column drop recommendations (>40% nulls)
null_pct = df.isna().mean() * 100
rec_df = pd.DataFrame({
    "Null %": null_pct.round(2),
    "Recommendation": null_pct.apply(
        lambda p: "DROP (>40%)" if p > 40 else "KEEP"
    )
}).sort_values("Null %", ascending=False)

to_drop = rec_df[rec_df["Recommendation"] == "DROP (>40%)"]
if not to_drop.empty:
    print("=" * 55)
    print(f"  {len(to_drop)} column(s) with >40% nulls — recommended for removal")
    print("=" * 55)
    display(to_drop)
else:
    print("No columns exceed 40% nulls — none recommended for removal.")

## 4. Numerical Columns – Descriptive Statistics

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numerical columns ({len(num_cols)}): {num_cols}")
print(f"Categorical / text columns ({len(cat_cols)}): {cat_cols}")

In [ ]:
if num_cols:
    display(df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T)

## 5. Categorical Columns – Value Counts

In [ ]:
for col in cat_cols[:10]:
    vc = df[col].value_counts(dropna=False).head(15)
    print(f"\n--- {col} ({vc.sum():,} non-null / {df[col].nunique():,} unique) ---")
    print(vc.to_string())

## 6. Outlier Detection (IQR method)

In [ ]:
if num_cols:
    outlier_data = []
    for col in num_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = df[(df[col] < lower) | (df[col] > upper)][col].dropna()
        if len(outliers) > 0:
            outlier_data.append({
                "Column": col,
                "Outliers": len(outliers),
                "%": round(len(outliers) / df[col].notna().sum() * 100, 2),
                "Lower bound": round(lower, 2),
                "Upper bound": round(upper, 2)
            })
    if outlier_data:
        outlier_df = pd.DataFrame(outlier_data).sort_values("Outliers", ascending=False)
        display(outlier_df)
    else:
        print("No outliers detected via IQR.")

In [ ]:
if num_cols:
    n_cols = min(6, len(num_cols))
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 4))
    if n_cols == 1:
        axes = [axes]
    for ax, col in zip(axes, num_cols[:n_cols]):
        sns.boxplot(y=df[col], ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

## 7. Correlation Analysis

In [ ]:
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    plt.figure(figsize=(max(8, len(num_cols) * 0.6), max(6, len(num_cols) * 0.5)))
    sns.heatmap(corr, mask=mask, annot=len(num_cols) <= 20,
                fmt=".2f", cmap="RdBu_r", center=0,
                square=True, linewidths=0.5)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()

    corr_unstacked = corr.unstack()
    corr_unstacked = corr_unstacked[
        corr_unstacked.index.get_level_values(0) !=
        corr_unstacked.index.get_level_values(1)
    ]
    top_corr = corr_unstacked.abs().sort_values(ascending=False).drop_duplicates().head(10)
    print("Top 10 strongest correlations:")
    for (col1, col2), val in top_corr.items():
        print(f"  {col1}  –  {col2}: {corr.loc[col1, col2]:+.4f}")

## 8. Distribution of Numerical Columns

In [ ]:
if num_cols:
    n_hist = min(9, len(num_cols))
    n_rows = (n_hist + 2) // 3
    fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4 * n_rows))
    axes = axes.flatten()
    for ax, col in zip(axes, num_cols[:n_hist]):
        sns.histplot(df[col].dropna(), kde=True, bins=40, ax=ax)
        ax.set_title(col)
    for ax in axes[n_hist:]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

## 9. Summary

In [ ]:
rows_with_nulls = (df.isna().sum(axis=1) >= 1).sum()
cols_to_drop = (df.isna().mean() * 100 > 40).sum()

print("=" * 60)
print("PRELIMINARY EDA SUMMARY")
print("=" * 60)

print(f"\nDataset: {file_path.name}")
print(f"Dimensions: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"Missing cells: {df.isna().sum().sum():,} / {df.size:,} ({df.isna().sum().sum() / df.size * 100:.2f}%)")
print(f"Rows with ≥1 null: {rows_with_nulls:,} ({rows_with_nulls / len(df) * 100:.1f}%)")
print(f"Columns with >40% nulls (recommended for removal): {cols_to_drop}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")